# Lab 6 · Climate and the fire season

**Day 3 · about 30 minutes · Student notebook**

> **Goal.** Chart a year of rainfall and temperature, and show why the fire season happens exactly when it does.

---

### How to work in this notebook

Each step gives you a **prompt card**. Copy it into Claude (or Gemini in Colab),
paste the code you get back into the empty cell below it, and run it.
Then read the **check** underneath and make sure your numbers are believable.

If the AI's code throws an error: copy the **whole** error message, paste it back to the
assistant with the sentence *"this is the error I got, please fix the code"*, and try again.
Do not retype the code by hand.


## Closing the loop

In Lab 5 you found that Chiang Mai burns between February and April. Now find out **why** —
and produce the chart that explains it to someone who has never seen a satellite image.

### Setup

In [ ]:
# Run this first, every session. Colab forgets everything when it recycles.
!pip install -q geemap

import ee, geemap

ee.Authenticate()                      # opens a link - sign in, paste the code back
ee.Initialize(project='YOUR-PROJECT-ID')   # <-- put YOUR project ID here

print('Earth Engine is ready.')

In [ ]:
PROVINCE = 'Chiang Mai'
aoi_fc = (ee.FeatureCollection('FAO/GAUL/2015/level1')
          .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
          .filter(ee.Filter.eq('ADM1_NAME', PROVINCE)))
aoi = aoi_fc.geometry()

### Step 1 — monthly rainfall

#### 🤖 Prompt card — Monthly rainfall for one year

Copy everything in the box into your AI assistant, then paste its answer into the empty cell below.

```text
PURPOSE   Get total rainfall for each month of 2024 in my province.
RESOURCE  Use UCSB-CHG/CHIRPS/DAILY, band "precipitation", units mm per day.
OUTLINE   Region is `aoi`. Year 2024, all twelve months.
MUST      For each month, SUM the daily images to get the monthly total,
          then take the spatial MEAN over the province. Use scale=5566.
          Return a plain Python list of 12 numbers.
PLATFORM  Google Colab, Earth Engine Python API.
TEST      Print the list, and print the driest and wettest month.
```

In [ ]:
# Paste the code your AI assistant gave you here, then run it.


> ### ✓ Check your answer
>
> Chiang Mai 2024: Jan **2.8 mm**, Feb 5.1, Mar 12.3, Apr 30.1, … Aug **318.6 mm**. The dry season is genuinely, almost absolutely dry — under 3 mm for a whole month.

### Step 2 — monthly temperature

In [ ]:
era5 = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
        .filterDate(f'{YEAR}-01-01', f'{YEAR+1}-01-01').select('temperature_2m'))

temp = []
for m in range(1, 13):
    start = ee.Date.fromYMD(YEAR, m, 1)
    k = (era5.filterDate(start, start.advance(1, 'month')).first()
         .reduceRegion(ee.Reducer.mean(), aoi, 11132,
                       maxPixels=1e9, bestEffort=True).getInfo()['temperature_2m'])
    temp.append(round(k - 273.15, 1))     # ERA5 is in KELVIN

for name, t in zip(MONTHS, temp):
    print(f'{name}  {t:>5.1f} C')

> ### ✓ Check your answer
>
> Peak is **April at 28.8 °C**, not June. The hottest month in northern Thailand is the end of the dry season, before the monsoon arrives and cools things down. If your numbers are around 300, you forgot to subtract 273.15.

### Step 3 — the chart that explains the fire season

#### 🤖 Prompt card — Rainfall and temperature on one chart

Copy everything in the box into your AI assistant, then paste its answer into the empty cell below.

```text
PURPOSE   Make one chart that shows why the forest burns in February to April.
RESOURCE  Two Python lists already exist: `rain` (12 monthly totals in mm) and
          `temp` (12 monthly means in Celsius). Month labels are in `MONTHS`.
OUTLINE   One figure, twelve months of 2024.
MUST      Rainfall as blue bars on the left axis, temperature as a line with markers on a
          twin right axis. Shade February to April to mark the fire season.
          Remove the top and right chart borders. Label both axes with units.
PLATFORM  Google Colab with matplotlib.
TEST      Show the chart inline.
```

In [ ]:
# Paste the code your AI assistant gave you here, then run it.


### Read the chart

Rainfall collapses to almost nothing in January. Temperature climbs to its annual peak in April.
For three months the forest is hot, bone dry, and — if it is deciduous — carpeted in dead leaves.

That is the fire season. You measured its extent in Lab 5 (**67,128 ha in 2024**), and now you
have the physical explanation on one page.

**This chart is the deliverable.** It is the thing a district officer or a journalist will
understand immediately, and you built it from two global datasets in about twenty lines.

> ### ✓ Check your answer
>
> **Compare with what you found in Lab 5:** dry season total (Jan–Mar) was **20.2 mm**;
> wet season (Jul–Sep) was **769.6 mm**. That is a **38-fold** difference. Now try
> changing `YEAR` to 2025 and see whether the lighter 2025 fire season
> (33,482 ha, half of 2024) shows up as a wetter or cooler dry season.